In [6]:
from py2neo import Graph
import json
import ast

In [7]:
# Connect to the database
REMOTE_HOST = "192.168.11.174"
graph = Graph(f"bolt://{REMOTE_HOST}:7687", auth=("neo4j", "admin123"))

try:
    version = graph.run("CALL dbms.components() YIELD versions RETURN versions[0]").evaluate()
    print(f"Connected to Neo4j version: {version}")
except Exception as e:
    print(f"Connection failed: {e}")

Connected to Neo4j version: 5.26.5


In [8]:
# EXTRACT DATA
print("Fetching graph data...")

query = """
MATCH (n:Entity) 
RETURN 
    n.id AS identifier, 
    n.label AS label,
    n.sem_types AS sem_types, 
    
    // Centrality & Community
    n.pageRank AS pagerank,
    n.authScore AS authscore,
    n.hubScore AS hubscore,
    n.communityId AS communityId,
    
    // Node2Vec (Required for thesis recreation)
    n.node2vec32_embedding AS node2vec32, 
    n.node2vec64_embedding AS node2vec64, 
    n.node2vec128_embedding AS node2vec128,
    
    // FastRP
    n.fastRP32_embedding AS fastrp32,
    n.fastRP64_embedding AS fastrp64,
    n.fastRP128_embedding AS fastrp128,
    
    // GraphSAGE
    n.graphSage32_embedding AS graphsage32,
    n.graphSage64_embedding AS graphsage64,
    n.graphSage128_embedding AS graphsage128,
    
    // HashGNN
    n.hashgnn32_embedding AS hashgnn32,
    n.hashgnn64_embedding AS hashgnn64,
    n.hashgnn128_embedding AS hashgnn128
"""



Fetching graph data...


In [9]:
result = graph.run(query)
data = {}

# Properties to save into JSON
properties_to_save = [
    "label", "pagerank", "authscore", "hubscore", "communityId",
    "node2vec32", "node2vec64", "node2vec128",
    "fastrp32", "fastrp64", "fastrp128",
    "graphsage32", "graphsage64", "graphsage128",
    "hashgnn32", "hashgnn64", "hashgnn128"
]

for record in result:
    cui = record["identifier"]
    
    # Safely handle 'sem_types'
    # If Neo4j stored it as a string like "[anim]", I parse it into a real Python list ["anim"].
    raw_sem_types = record["sem_types"]
    if isinstance(raw_sem_types, str):
        try:
            sem_types_list = ast.literal_eval(raw_sem_types)
        except:
            sem_types_list = [raw_sem_types.strip("[]")]
    else:
        sem_types_list = raw_sem_types or []

    # Build the dictionary for this node
    node_data = {"sem_types": sem_types_list}
    for prop in properties_to_save:
        node_data[prop] = record[prop]
        
    data[cui] = node_data

In [10]:
# SAVE TO JSON
print(f"Exporting {len(data)} nodes to fulldb-entities.json...")

with open('fulldb-entities.json', 'w') as json_file:
    json.dump(data, json_file, sort_keys=True, indent=4)

print("Data Acquisition Complete!")

Exporting 33724 nodes to fulldb-entities.json...
Data Acquisition Complete!


2',3'-Cyclic-Nucleotide Phosphodiesterases